# 04 — LLM to SQL: question → SQL → validate → run read-only

Groq (`openai/gpt-oss-120b`) writes SQL from a description of the data (never the rows).
Safety is enforced in **code**, in three layers, so it holds even if the LLM is tricked:

| Layer | What it stops |
|---|---|
| 1. `validate_sql()` with sqlglot | anything but ONE `SELECT` on known tables/columns; adds `LIMIT` |
| 2. SQLite authorizer | any action except reading, even if layer 1 missed something |
| 3. Read-only connection + timeout | writes at the file level; runaway queries |

## Section 0 — Setup


In [1]:
import json
import os
import sqlite3
import time
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import sqlglot
from dotenv import load_dotenv
from groq import Groq
from sqlglot import exp

In [2]:
PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "processed" / "chatbot.db"
GROQ_MODEL = "openai/gpt-oss-120b"
MAX_ROWS = 200            # never return more rows than this
QUERY_TIMEOUT_S = 5       # stop any query that runs longer
CATEGORY_LIMIT = 25       # list text columns with this many distinct values or fewer

load_dotenv(PROJECT_ROOT / ".env")
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("DB exists:", DB_PATH.exists(), "| model:", GROQ_MODEL)


DB exists: True | model: openai/gpt-oss-120b


## Section 1 — Schema from the catalog
The LLM sees this text instead of the data. `load_schema()` gives the validator the list of
allowed tables and columns.


In [3]:
def connect_db_read_only(path):
    conn = sqlite3.connect(f"{Path(path).resolve().as_uri()}?mode=ro", uri=True)
    conn.row_factory = sqlite3.Row
    return conn


def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'


def load_schema(db_path):
    """{table: {column: data_type}} for every imported table, read from the catalog."""
    conn = connect_db_read_only(db_path)
    try:
        rows = conn.execute(
            "SELECT table_name, column_name, data_type FROM _catalog_columns ORDER BY table_name, position"
        ).fetchall()
    finally:
        conn.close()
    schema = {}
    for row in rows:
        schema.setdefault(row["table_name"], {})[row["column_name"]] = row["data_type"]
    return schema


def describe_schema(db_path):
    """Plain-text description of every table: what the LLM sees instead of the data."""
    conn = connect_db_read_only(db_path)
    try:
        lines = []
        for t in conn.execute("SELECT * FROM _catalog_tables ORDER BY table_name").fetchall():
            table = t["table_name"]
            lines.append(f"Table {table}: {t['row_count']} rows, from '{t['source_file']}'")
            lines.append("- _row_id integer (row number in the source file)")
            for c in conn.execute("SELECT * FROM _catalog_columns WHERE table_name = ? ORDER BY position", (table,)):
                line = f"- {c['column_name']} {c['data_type']}"
                if c["null_count"]:
                    line += f", {c['null_count']} blank"
                if c["min_value"] is not None:
                    line += f", range {c['min_value']}..{c['max_value']}"
                if c["data_type"] == "text" and c["distinct_count"] <= CATEGORY_LIMIT:
                    name = quote_identifier(c["column_name"])
                    values = [r[0] for r in conn.execute(
                        f"SELECT DISTINCT {name} FROM {quote_identifier(table)} WHERE {name} IS NOT NULL ORDER BY 1")]
                    line += f", values: {', '.join(repr(v) for v in values)}"
                elif c["data_type"] == "text":
                    line += f", free text (avg {c['avg_length']:.0f} chars)"
                lines.append(line)
        return "\n".join(lines)
    finally:
        conn.close()


schema = load_schema(DB_PATH)
schema_text = describe_schema(DB_PATH)
print(schema_text)


Table reviews: 23486 rows, from 'Womens Clothing E-Commerce Reviews.csv'
- _row_id integer (row number in the source file)
- review_id integer, range 0..23485
- clothing_id integer, range 0..1205
- age integer, range 18..99
- title text, 3810 blank, free text (avg 19 chars)
- review_text text, 845 blank, free text (avg 309 chars)
- rating integer, range 1..5
- recommended_ind integer, range 0..1
- positive_feedback_count integer, range 0..122
- division_name text, 14 blank, values: 'General', 'General Petite', 'Initmates'
- department_name text, 14 blank, values: 'Bottoms', 'Dresses', 'Intimate', 'Jackets', 'Tops', 'Trend'
- class_name text, 14 blank, values: 'Blouses', 'Casual bottoms', 'Chemises', 'Dresses', 'Fine gauge', 'Intimates', 'Jackets', 'Jeans', 'Knits', 'Layering', 'Legwear', 'Lounge', 'Outerwear', 'Pants', 'Shorts', 'Skirts', 'Sleep', 'Sweaters', 'Swim', 'Trend'


## Section 2 — Layer 1: validate the SQL with sqlglot
sqlglot parses SQL into a tree, so we check what the query **is**, not what its text looks like.
Allow-list: exactly one SELECT (or UNION/INTERSECT/EXCEPT of SELECTs), known tables and columns only.


In [4]:
class SQLValidationError(Exception):
    """The SQL is not a single, safe, read-only SELECT on known tables and columns."""


ALLOWED_ROOTS = (exp.Select, exp.Union, exp.Intersect, exp.Except)
FORBIDDEN_NODES = (
    exp.Insert, exp.Update, exp.Delete, exp.Drop, exp.Create, exp.Alter, exp.Merge,
    exp.Command, exp.Pragma, exp.Attach, exp.Detach, exp.Transaction, exp.Commit,
    exp.Rollback, exp.Into, exp.Set, exp.Use, exp.Copy, exp.LoadData,
)
BLOCKED_FUNCTIONS = {"load_extension", "readfile", "writefile", "edit", "fts3_tokenizer"}


def validate_sql(sql, schema, max_rows=MAX_ROWS):
    """Return a safe version of `sql` (with a LIMIT) or raise SQLValidationError explaining why not."""
    try:
        statements = [s for s in sqlglot.parse(sql, read="sqlite") if s is not None]
    except sqlglot.errors.ParseError as exc:
        raise SQLValidationError(f"Could not parse the SQL: {exc}") from exc
    if len(statements) != 1:
        raise SQLValidationError(f"Expected exactly one statement, got {len(statements)}")
    tree = statements[0]

    if not isinstance(tree, ALLOWED_ROOTS):
        raise SQLValidationError(f"Only SELECT queries are allowed, got {type(tree).__name__.upper()}")
    for node in tree.walk():
        if isinstance(node, FORBIDDEN_NODES):
            raise SQLValidationError(f"Forbidden operation inside the query: {type(node).__name__.upper()}")
    for function in tree.find_all(exp.Anonymous):
        if function.name.lower() in BLOCKED_FUNCTIONS:
            raise SQLValidationError(f"Function '{function.name}' is not allowed")

    cte_names = {cte.alias_or_name for cte in tree.find_all(exp.CTE)}
    tables = {t.name for t in tree.find_all(exp.Table)} - cte_names
    unknown_tables = sorted(tables - schema.keys())
    if unknown_tables:
        raise SQLValidationError(f"Unknown table(s): {unknown_tables}. Allowed: {sorted(schema)}")

    known_columns = {"_row_id"}.union(*(schema[t].keys() for t in tables))
    aliases = {a.alias for a in tree.find_all(exp.Alias)}
    for column in tree.find_all(exp.Column):
        if column.name not in known_columns | aliases:
            raise SQLValidationError(
                f"Unknown column '{column.name}'. Text values need single quotes, e.g. department_name = 'Tops'")

    if isinstance(tree, exp.Select):
        limit = tree.args.get("limit")
        current = limit.expression if limit is not None else None
        if not (isinstance(current, exp.Literal) and current.is_int and int(current.this) <= max_rows):
            tree = tree.limit(max_rows)
    else:
        tree = exp.select("*").from_(tree.subquery("result")).limit(max_rows)
    return tree.sql(dialect="sqlite")


## Section 3 — Layers 2 and 3: run it safely


In [5]:
ALLOWED_ACTIONS = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION, sqlite3.SQLITE_RECURSIVE}


def read_only_authorizer(action, arg1, arg2, db_name, trigger):
    """SQLite asks this before every step of a query; anything but reading is denied."""
    if action == sqlite3.SQLITE_FUNCTION and (arg2 or "").lower() in BLOCKED_FUNCTIONS:
        return sqlite3.SQLITE_DENY
    return sqlite3.SQLITE_OK if action in ALLOWED_ACTIONS else sqlite3.SQLITE_DENY


def run_select(db_path, sql, timeout_s=QUERY_TIMEOUT_S):
    """Run a validated SELECT on a read-only connection, with an authorizer and a time limit."""
    conn = connect_db_read_only(db_path)
    conn.set_authorizer(read_only_authorizer)
    deadline = time.monotonic() + timeout_s
    conn.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, 10_000)
    try:
        cursor = conn.execute(sql)
        columns = [d[0] for d in cursor.description]
        rows = [tuple(r) for r in cursor.fetchall()]
    finally:
        conn.close()
    return pd.DataFrame(rows, columns=columns)


### Check: safe SQL passes, unsafe SQL is rejected — without any LLM


In [6]:
safe = [
    "SELECT department_name, AVG(rating) AS avg_rating FROM reviews GROUP BY department_name",
    "WITH neg AS (SELECT * FROM reviews WHERE rating <= 2) SELECT class_name, COUNT(*) AS n FROM neg GROUP BY class_name",
    "SELECT rating FROM reviews LIMIT 100000",
    "SELECT review_id FROM reviews WHERE rating = 1 UNION SELECT review_id FROM reviews WHERE rating = 5",
]
for sql in safe:
    print("OK  ->", validate_sql(sql, schema))

unsafe = [
    "SELECT 1; DROP TABLE reviews",
    "DELETE FROM reviews",
    "PRAGMA table_info(reviews)",
    "ATTACH DATABASE 'x.db' AS x",
    "SELECT load_extension('evil')",
    "SELECT * FROM sqlite_master",
    "SELECT * FROM _catalog_tables",
    "SELECT secret FROM reviews",
    'SELECT * FROM reviews WHERE department_name = "Tops"',
    "VACUUM",
]
for sql in unsafe:
    try:
        validate_sql(sql, schema)
        print("UNEXPECTED: accepted ->", sql)
    except SQLValidationError as exc:
        print(f"Rejected: {sql[:40]:<40} | {exc}")


OK  -> SELECT department_name, AVG(rating) AS avg_rating FROM reviews GROUP BY department_name LIMIT 200
OK  -> WITH neg AS (SELECT * FROM reviews WHERE rating <= 2) SELECT class_name, COUNT(*) AS n FROM neg GROUP BY class_name LIMIT 200
OK  -> SELECT rating FROM reviews LIMIT 200
OK  -> SELECT * FROM (SELECT review_id FROM reviews WHERE rating = 1 UNION SELECT review_id FROM reviews WHERE rating = 5) AS result LIMIT 200
Rejected: SELECT 1; DROP TABLE reviews             | Expected exactly one statement, got 2
Rejected: DELETE FROM reviews                      | Only SELECT queries are allowed, got DELETE
Rejected: PRAGMA table_info(reviews)               | Only SELECT queries are allowed, got PRAGMA
Rejected: ATTACH DATABASE 'x.db' AS x              | Only SELECT queries are allowed, got ATTACH
Rejected: SELECT load_extension('evil')            | Function 'load_extension' is not allowed
Rejected: SELECT * FROM sqlite_master              | Unknown table(s): ['sqlite_master']. Allowed: 

In [7]:
print(run_select(DB_PATH, "SELECT COUNT(*) AS n FROM reviews"))

# Bypass the validator on purpose: layers 2 and 3 must still refuse
for sql in ["DELETE FROM reviews", "SELECT load_extension('x')", "CREATE TEMP TABLE t(x)"]:
    try:
        run_select(DB_PATH, sql)
        print("UNEXPECTED: ran ->", sql)
    except sqlite3.Error as exc:
        print(f"Blocked at runtime: {sql:<30} | {type(exc).__name__}: {exc}")

# A query that never ends is stopped by the timeout
endless = "WITH RECURSIVE c(x) AS (SELECT 1 UNION ALL SELECT x + 1 FROM c) SELECT COUNT(*) FROM c"
try:
    run_select(DB_PATH, endless, timeout_s=1)
except sqlite3.Error as exc:
    print("Stopped after 1s ->", exc)


       n
0  23486
Blocked at runtime: DELETE FROM reviews            | DatabaseError: not authorized
Blocked at runtime: SELECT load_extension('x')     | OperationalError: not authorized to use function: load_extension
Blocked at runtime: CREATE TEMP TABLE t(x)         | OperationalError: not authorized
Stopped after 1s -> interrupted
